In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/image-differential-annotator/')

from tqdm import tqdm
from annotator.combineCDF import getDiscreteCombinedCDFofAllFeatures as PCMA
from annotator.stqutils import loadAd, preparePatchesWSI, getPatchRepresentation, inferProb, showProb, showProbImg
from annotator.stqutils import trainClassifier, showMolecularData

import zarr
import tifffile
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

qs = np.linspace(0.05, 0.95, 10, endpoint=True)

In [31]:
samples = ['JDC-WP-001-s', 'JDC-WP-001-c', 'JDC-WP-002-r', 'JDC-WP-002-c', 'JDC-WP-003-ae', 'JDC-WP-003-d', 'JDC-WP-004-y', 'JDC-WP-004-c',
       'JDC-WP-005-n', 'JDC-WP-005-c', 'JDC-WP-007-o', 'JDC-WP-007-c', 'JDC-WP-008-v', 'JDC-WP-008-b', 'JDC-WP-009-n', 'JDC-WP-009-c',
       'JDC-WP-010-w', 'JDC-WP-010-c', 'JDC-WP-011-b', 'JDC-WP-011-w', 'JDC-WP-002-v', 'JDC-WP-012-ae', 'JDC-WP-012-c', 'JDC-WP-012-w',
       'JDC-WP-013-y', 'JDC-WP-013-b', 'JDC-WP-014-aj', 'JDC-WP-014-d', 'JDC-WP-015-n', 'JDC-WP-015-b', 'JDC-WP-016-x', 'JDC-WP-016-b']

mpp = 0.25
ts = 56.

In [7]:
anatomical_location = {a: b for b, a in \
                       [['Head-sup', 'JDC-WP-001-s'], # 001
                        ['Head-inf', 'JDC-WP-001-x'],
                        ['Body',     'JDC-WP-001-l'],
                        ['Tail',     'JDC-WP-001-c'],
                        ['Head-sup', 'JDC-WP-002-r'], # 002
                        ['Head-inf', 'JDC-WP-002-v'],
                        ['Body',     'JDC-WP-002-j'],
                        ['Tail',     'JDC-WP-002-c'],
                        ['Head-sup', 'JDC-WP-003-ae'], # 003
                        ['Head-inf', 'JDC-WP-003-aq'],
                        ['Body',     'JDC-WP-003-r'],
                        ['Tail',     'JDC-WP-003-d'],
                        ['Head-sup', 'JDC-WP-004-y'], # 004
                        ['Head-inf', 'JDC-WP-004-ah'],                       
                        ['Body',     'JDC-WP-004-n'],
                        ['Tail',     'JDC-WP-004-c'],
                        ['Head-sup', 'JDC-WP-005-n'], # 005
                        ['Head-inf', 'JDC-WP-005-r'],
                        ['Body',     'JDC-WP-005-j'],
                        ['Tail',     'JDC-WP-005-c'],
                        ['Head-sup', 'JDC-WP-006-m'], # 006
                        ['Head-inf', 'JDC-WP-006-t'],
                        ['Body',     'JDC-WP-006-h'],
                        ['Tail',     'JDC-WP-006-c'],
                        ['Head-sup', 'JDC-WP-007-o'], # 007
                        ['Head-inf', 'JDC-WP-007-s'],
                        ['Body',     'JDC-WP-007-j'],
                        ['Tail',     'JDC-WP-007-c'],
                        ['Head-sup', 'JDC-WP-008-r'], # 008
                        ['Head-inf', 'JDC-WP-008-v'],
                        ['Body',     'JDC-WP-008-j'],
                        ['Tail',     'JDC-WP-008-b'],
                        ['Head-sup', 'JDC-WP-009-n'], # 009
                        ['Head-inf', 'JDC-WP-009-r'],
                        ['Body',     'JDC-WP-009-j'],
                        ['Tail',     'JDC-WP-009-c'],
                        ['Head-sup', 'JDC-WP-010-w'], # 010
                        ['Head-inf', 'JDC-WP-010-ac'],
                        ['Body',     'JDC-WP-010-p'],
                        ['Tail',     'JDC-WP-010-c'],
                        ['Head-sup', 'JDC-WP-011-w'], # 011
                        ['Head-inf', 'JDC-WP-011-ac'],
                        ['Body',     'JDC-WP-011-n'],
                        ['Tail',     'JDC-WP-011-b'],
                        ['Head-sup', 'JDC-WP-012-w'], # 012
                        ['Head-inf', 'JDC-WP-012-ae'],
                        ['Body',     'JDC-WP-012-n'],
                        ['Tail',     'JDC-WP-012-c'],
                        ['Head-sup', 'JDC-WP-013-y'], # 013
                        ['Head-inf', 'JDC-WP-013-ag'],
                        ['Body',     'JDC-WP-013-n'],
                        ['Tail',     'JDC-WP-013-b'],
                        ['Head-sup', 'JDC-WP-014-aj'], # 014
                        ['Head-inf', 'JDC-WP-014-ap'],
                        ['Body',     'JDC-WP-014-r'],
                        ['Tail',     'JDC-WP-014-d'],
                        ['Head-sup', 'JDC-WP-015-n'], # 015
                        ['Head-inf', 'JDC-WP-015-u'],
                        ['Body',     'JDC-WP-015-h'],
                        ['Tail',     'JDC-WP-015-b'],
                        ['Head-sup', 'JDC-WP-016-x'], # 016
                        ['Head-inf', 'JDC-WP-016-ag'],
                        ['Body',     'JDC-WP-016-n'],
                        ['Tail',     'JDC-WP-016-b'],
                       ]}

ages = {'JDC-WP-001': 21,
        'JDC-WP-002': 71,
        'JDC-WP-003': 32,
        'JDC-WP-004': 54,
        'JDC-WP-005': 28,
        'JDC-WP-006': 62,
        'JDC-WP-007': 22,
        'JDC-WP-008': 23,
        'JDC-WP-009': 40,
        'JDC-WP-010': 37,
        'JDC-WP-011': 47,
        'JDC-WP-012': 69,
        'JDC-WP-013': 35,
        'JDC-WP-014': 39,
        'JDC-WP-015': 53,
        'JDC-WP-016': 43}

age_groups = {}
for case in ages.keys():
    if ages[case] < 33:
        g = 'young'
    elif (ages[case] >= 33) and (ages[case] < 60):
        g = 'middle'
    else:
        g = 'old'    
    age_groups[case] = g

In [ ]:
head_tissues = [s for s in samples if anatomical_location[s].startswith('Head')]
tail_tissues = [s for s in samples if anatomical_location[s].startswith('Tail')]
print(len(head_tissues), len(tail_tissues))

old_tissues = [s for s in samples if age_groups[s[:10]] == 'old']
young_tissues = [s for s in samples if age_groups[s[:10]] == 'young']
print(len(old_tissues), len(young_tissues))

In [ ]:
outsSTQPath = '/projects/activities/kappsen-tmc/USERS/domans/results-STQ-post-xenium-32-final/'
df_ids = pd.read_csv('/sdata/activities/kappsen-tmc/visium/analysis/downstream/Pancreas Xenium identifiers and paths.csv', index_col=0)
IDsToSTQnames = df_ids['HE-slide-identifier'].to_dict()

ads = {sample: loadAd(outsSTQPath + IDsToSTQnames[sample] + '/', L=None, fname='img.data.ctranspath-1.h5ad', suffix=None)[0] for sample in tqdm(samples)}

In [ ]:
# Prepare the patches coordinates for each sample and concatenate them into a single DataFrame
patchCoordinates = pd.concat([preparePatchesWSI(ads[sample].obs, N=8, spacing=56/0.25, sample_id=sample) for sample in tqdm(samples)], axis=0)

# Get the patch SAMPLER representations for each sample and combine them into a single DataFrame
patchesCDFs = pd.concat([getPatchRepresentation(ads[sample], patchCoordinates.xs(sample, level='sample', axis=0), qs, sample_id=sample) for sample in tqdm(samples)], axis=0)

In [ ]:
if True:
    positive_patches = patchesCDFs.index[patchesCDFs.index.get_level_values('sample').isin(old_tissues)]
    negative_patches = patchesCDFs.index[patchesCDFs.index.get_level_values('sample').isin(young_tissues)]
else:
    positive_patches = patchesCDFs.index[patchesCDFs.index.get_level_values('sample').isin(head_tissues)]
    negative_patches = patchesCDFs.index[patchesCDFs.index.get_level_values('sample').isin(tail_tissues)]

print('Number of positive patches:', positive_patches.shape)
print('Number of negative patches:', negative_patches.shape)

annotation_results = {}
annotation_results.update({p: 'positive' for p in positive_patches})
annotation_results.update({p: 'negative' for p in negative_patches})

In [89]:
alpha = 0.65
clf = trainClassifier(annotation_results, patchesCDFs, alpha=alpha, seed=0, augFunc=PCMA)

In [ ]:
for id in ['JDC-WP-002-v', 'JDC-WP-002-c', 'JDC-WP-008-v', 'JDC-WP-008-b']:
    x, y, p = inferProb(ads[id], clf, qs, tsize=56/0.25, R=2)
    showProbImg(x, y, p, f=1, figsize=(5, 3), ts=ts, mpp=mpp, title=id  + r', PCMA $\alpha$=' + str(alpha), filter=0.5)
    search = ads[id].obs.set_index(['pxl_col_in_wsi', 'pxl_row_in_wsi'])['original_barcode']
    mi = pd.MultiIndex.from_arrays([x, y], names=['pxl_col_in_wsi', 'pxl_row_in_wsi'])
    pindex = search.loc[mi].values
    se_inf = pd.Series(index=pindex, data=p)